# 06 — Experimento 2: resultados

Compara, sobre las mismas ventanas y con tu misma referencia, las respuestas de cada modelo de lenguaje en las 4 variantes de la prueba (N0 y N1, con señales en filas y con reportes en filas). **No necesita GPU:** lee las ejecuciones guardadas por `05_experimento_2`.

**Criterio (el de la memoria):** anomalía = 1; normal y anomalía mitigada = 0; las dudosas quedan fuera del cálculo principal y se cuentan aparte de las dos formas. Una ventana sin veredicto (JSON no válido o respuesta cortada) cuenta como fallo.

**Comparación principal, fijada antes de ejecutar la prueba (12/10/2026):** ¿acierta Qwen3.5-9B más que Qwen2.5-7B en N1 con señales en filas, sobre las 40 ventanas de prueba? Prueba exacta de McNemar, sin las dudosas. El resto de modelos y variantes se presentan como resultados descriptivos.

| Sección | Qué hace |
|---|---|
| 1–3 | Entorno, parámetros y carga de las ejecuciones |
| 4 | Resultados: comparación principal, métricas, grupos, N0 frente a N1, pares de modelos, dudosas y tabla por ventana |
| 5 | Revisión: una ventana (gráfica y veredictos) y el informe HTML con las respuestas (privado) |
| 6 | Guardado de las cifras en `resultados/experimento_2/` |

**Confidencial:** el notebook solo muestra cifras, veredictos y nombres cortos. Las respuestas del modelo solo aparecen en el informe HTML, que se guarda en `privado/`.

## 1. Entorno

En Colab, monta Google Drive y clona el repositorio para tener acceso a `src/`. En local, busca la carpeta `src/` subiendo desde el directorio actual.

In [1]:
# --- Entorno: localiza src/ y, en Colab, monta Google Drive ---
import os
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules
URL_REPOSITORIO = "https://github.com/Williams-MP/TFM_VIU_2026.git"
RUTA_REPO_COLAB = "/content/TFM_VIU_2026"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(RUTA_REPO_COLAB):
        # Mientras el repositorio sea privado hace falta un token de GitHub.
        # Si es público, basta con dejar el campo vacío.
        from getpass import getpass
        token = getpass("Token de GitHub (vacío si el repositorio es público): ")
        url = URL_REPOSITORIO.replace("https://", f"https://{token}@") if token else URL_REPOSITORIO
        subprocess.run(["git", "clone", "-q", url, RUTA_REPO_COLAB], check=True)
    ruta_src = f"{RUTA_REPO_COLAB}/src"
else:
    # Ejecución local: se sube por las carpetas hasta encontrar src/.
    directorio = os.getcwd()
    while not os.path.isdir(os.path.join(directorio, "src")):
        padre = os.path.dirname(directorio)
        if padre == directorio:
            raise FileNotFoundError("No se encuentra la carpeta src/ del repositorio.")
        directorio = padre
    ruta_src = os.path.join(directorio, "src")

if ruta_src not in sys.path:
    sys.path.append(ruta_src)

import json
from datetime import datetime

import pandas as pd

from config import (RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP, NOMBRES_CONTADORES,
                    RUTA_EXPERIMENTO_2_PRIVADO, RUTA_EXPERIMENTO_2)
from llm_lab import exp2
from llm_lab.prompts import leer_correspondencia
from llm_lab.ventanas import FuenteVentanas

RUTA_ENTRADAS = f"{RUTA_EXPERIMENTO_2_PRIVADO}/entradas"
RUTA_EJECUCIONES_37 = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones_37"
RUTA_EJECUCIONES = f"{RUTA_EXPERIMENTO_2_PRIVADO}/ejecuciones"
RUTA_INFORMES = f"{RUTA_EXPERIMENTO_2_PRIVADO}/informes"     # HTML con las respuestas del modelo (privado)

fuente = FuenteVentanas(RUTA_PREPARADOS, IDX_CONTADORES, IDX_GAP)

Mounted at /content/drive


## 2. Parámetros

In [21]:
# --- Parámetros (EDITABLE) ---
CONJUNTO = "test"     # "test" (40 ventanas de prueba) o "dev" (29 de desarrollo)
EJECUCIONES = None    # None = la más reciente de cada modelo en privado/experimento_2/ejecuciones; o {modelo: base}

# Qwen2.5-7B: las ejecuciones de la memoria
REFERENCIA_QWEN25 = {"dev": f"{RUTA_EJECUCIONES_37}/fase1_20261008_1046.csv",
                     "test": f"{RUTA_EJECUCIONES_37}/fase2_20261008_1127.csv"}

# Comparación principal fijada antes de la prueba
PRINCIPAL = {"modelo_a": "qwen2.5-7b", "modelo_b": "qwen3.5-9b", "metodo": "N1", "orientacion": "senales_en_filas"}

## 3. Carga

Se avisa si alguna ejecución está incompleta (menos de 4 variantes × ventanas): puede ser una que se cortó y no se reanudó.

In [22]:
# --- Ventanas con tu referencia y ejecuciones de cada modelo ---
VENTANAS = exp2.cargar_ventanas(f"{RUTA_ENTRADAS}/{'ventanas_desarrollo' if CONJUNTO == 'dev' else 'ventanas_prueba'}.csv")
ejecuciones = EJECUCIONES or exp2.ultimas_ejecuciones(RUTA_EJECUCIONES, CONJUNTO)

partes = [exp2.cargar(REFERENCIA_QWEN25[CONJUNTO], "qwen2.5-7b", VENTANAS)]
for modelo, base in ejecuciones.items():
    partes.append(exp2.cargar(f"{RUTA_EJECUCIONES}/{base}.csv", modelo, VENTANAS))
    esperadas = 4 * len(VENTANAS)
    aviso = "" if len(partes[-1]) == esperadas else f"  AVISO: incompleta ({len(partes[-1])} de {esperadas} llamadas)"
    print(f"{modelo}: {base}{aviso}")
df = pd.concat(partes, ignore_index=True)
MODELOS = list(dict.fromkeys(df["modelo"]))
n_eval = sum(exp2.MAPEO_REFERENCIA.get(w["referencia"]) is not None for w in VENTANAS)
n_anom = sum(w["referencia"] == "anomalia" for w in VENTANAS)
print(f"{CONJUNTO}: {len(VENTANAS)} ventanas ({n_eval} sin dudosas) | modelos: {MODELOS}")
print(f"Referencia trivial «siempre anomalía»: {n_anom}/{n_eval} ({n_anom / n_eval:.1%})")

qwen2.5-14b: exp2_test_qwen2.5-14b_20261012_214454
qwen3.5-9b: exp2_test_qwen3.5-9b_20261012_204121
test: 40 ventanas (37 sin dudosas) | modelos: ['qwen2.5-7b', 'qwen2.5-14b', 'qwen3.5-9b']
Referencia trivial «siempre anomalía»: 25/37 (67.6%)


## 4. Resultados

In [23]:
# --- 4.1 Comparación principal (fijada antes de la prueba) ---
tabla_principal = exp2.comparacion_principal(df, **PRINCIPAL)
display(tabla_principal)
if CONJUNTO != "test":
    print("Nota: en desarrollo es solo orientativa; la comparación principal se decide en la prueba.")

,metodo,orientacion,ventanas,aciertos_qwen2.5-7b,aciertos_qwen3.5-9b,solo_qwen2.5-7b,solo_qwen3.5-9b,p_mcnemar,modelo_a,modelo_b
0,N1,senales_en_filas,37,26,29,1,4,0.375,qwen2.5-7b,qwen3.5-9b


In [24]:
# --- 4.2 Métricas por modelo y variante, y acierto de un vistazo (modelos en filas, variantes en columnas) ---
tabla_metricas = exp2.metricas(df)
display(tabla_metricas)
tabla_acierto = (tabla_metricas.assign(variante=tabla_metricas["metodo"] + " | " +
                                       tabla_metricas["orientacion"].str.replace("_en_filas", ""))
                 .pivot(index="modelo", columns="variante", values="aciertos").loc[MODELOS])
display(tabla_acierto)

,modelo,metodo,orientacion,ventanas,json_valido,cortadas,evaluables,sin_veredicto,aciertos,acierto,sensibilidad,especificidad,auroc_p_interna,auroc_n
0,qwen2.5-7b,N0,senales_en_filas,40,1.0,0,37,0,26,0.703,0.92,0.250,0.735,37
1,qwen2.5-7b,N0,reportes_en_filas,40,1.0,0,37,0,17,0.459,0.68,0.000,0.490,37
2,qwen2.5-7b,N1,senales_en_filas,40,1.0,0,37,0,26,0.703,1.00,0.083,0.653,37
3,qwen2.5-7b,N1,reportes_en_filas,40,1.0,0,37,0,22,0.595,0.72,0.333,0.660,37
4,qwen2.5-14b,N0,senales_en_filas,40,1.0,0,37,0,17,0.459,0.48,0.417,0.457,37
5,qwen2.5-14b,N0,reportes_en_filas,40,1.0,0,37,0,21,0.568,0.60,0.500,0.603,37
6,qwen2.5-14b,N1,senales_en_filas,40,1.0,0,37,0,27,0.730,0.92,0.333,0.808,37
7,qwen2.5-14b,N1,reportes_en_filas,40,1.0,0,37,0,27,0.730,0.84,0.500,0.857,37
8,qwen3.5-9b,N0,senales_en_filas,40,1.0,0,37,0,26,0.703,0.80,0.500,0.773,37
9,qwen3.5-9b,N0,reportes_en_filas,40,1.0,0,37,0,22,0.595,0.72,0.333,0.593,37


variante,N0 | reportes,N0 | senales,N1 | reportes,N1 | senales
modelo,,,,
qwen2.5-7b,17,26,22,26
qwen2.5-14b,21,17,27,27
qwen3.5-9b,22,26,28,29


In [25]:
# --- 4.3 Acierto por grupo de ventanas (con alerta, alta actividad sin alerta, al azar) ---
tabla_grupos = exp2.metricas_por_grupo(df)
display(tabla_grupos.pivot_table(index=["modelo", "metodo", "orientacion"], columns="grupo",
                                 values="aciertos", aggfunc="first"))

grupo                                 alerta_consenso_comun  \
modelo      metodo orientacion                                
qwen2.5-14b N0     reportes_en_filas                     15   
                   senales_en_filas                      12   
            N1     reportes_en_filas                     19   
                   senales_en_filas                      19   
qwen2.5-7b  N0     reportes_en_filas                     17   
                   senales_en_filas                      17   
            N1     reportes_en_filas                     18   
                   senales_en_filas                      19   
qwen3.5-9b  N0     reportes_en_filas                     18   
                   senales_en_filas                      17   
            N1     reportes_en_filas                     18   
                   senales_en_filas                      18   

grupo                                 control_alta_actividad  control_azar  
modelo      metodo orientacion                                              
qwen2.5-14b N0     reportes_en_filas                       2             4  
                   senales_en_filas                        2             3  
            N1     reportes_en_filas                       2             6  
                   senales_en_filas                        4             4  
qwen2.5-7b  N0     reportes_en_filas                       0             0  
                   senales_en_filas                        7             2  
            N1     reportes_en_filas                       1             3  
                   senales_en_filas                        7             0  
qwen3.5-9b  N0     reportes_en_filas                       2             2  
                   senales_en_filas                        5             4  
            N1     reportes_en_filas                       7             3  
                   senales_en_filas                        8             3

In [26]:
# --- 4.4 N0 frente a N1 en cada modelo, cada par de modelos en cada variante y efecto de las dudosas ---
tabla_n0_n1 = exp2.n0_frente_a_n1(df)
tabla_pares = exp2.todos_pareados(df, MODELOS)
tabla_dudosas = exp2.dudosas(df)
display(tabla_n0_n1, tabla_pares, tabla_dudosas)

,modelo,orientacion,ventanas,aciertos_N0,aciertos_N1,solo_N0,solo_N1,p_mcnemar
0,qwen2.5-7b,senales_en_filas,37,26,26,3,3,1.0000
1,qwen2.5-7b,reportes_en_filas,37,17,22,1,6,0.1250
2,qwen2.5-14b,senales_en_filas,37,17,27,4,14,0.0309
3,qwen2.5-14b,reportes_en_filas,37,21,27,4,10,0.1796
4,qwen3.5-9b,senales_en_filas,37,26,29,3,6,0.5078
5,qwen3.5-9b,reportes_en_filas,37,22,28,3,9,0.1460


,modelo_a,modelo_b,metodo,orientacion,ventanas,aciertos_a,aciertos_b,solo_a,solo_b,p_mcnemar
0,qwen2.5-7b,qwen2.5-14b,N0,senales_en_filas,37,26,17,13,4,0.0490
1,qwen2.5-7b,qwen2.5-14b,N0,reportes_en_filas,37,17,21,3,7,0.3438
2,qwen2.5-7b,qwen2.5-14b,N1,senales_en_filas,37,26,27,3,4,1.0000
3,qwen2.5-7b,qwen2.5-14b,N1,reportes_en_filas,37,22,27,1,6,0.1250
4,qwen2.5-7b,qwen3.5-9b,N0,senales_en_filas,37,26,26,4,4,1.0000
5,qwen2.5-7b,qwen3.5-9b,N0,reportes_en_filas,37,17,22,0,5,0.0625
6,qwen2.5-7b,qwen3.5-9b,N1,senales_en_filas,37,26,29,1,4,0.3750
7,qwen2.5-7b,qwen3.5-9b,N1,reportes_en_filas,37,22,28,3,9,0.1460
8,qwen2.5-14b,qwen3.5-9b,N0,senales_en_filas,37,17,26,2,11,0.0225
9,qwen2.5-14b,qwen3.5-9b,N0,reportes_en_filas,37,21,22,3,4,1.0000


,modelo,metodo,orientacion,sin_dudosas,dudosas_anomalia,dudosas_normal
0,qwen2.5-7b,N0,senales_en_filas,26/37,29/40,26/40
1,qwen2.5-7b,N0,reportes_en_filas,17/37,20/40,17/40
2,qwen2.5-7b,N1,senales_en_filas,26/37,27/40,28/40
3,qwen2.5-7b,N1,reportes_en_filas,22/37,25/40,22/40
4,qwen2.5-14b,N0,senales_en_filas,17/37,19/40,18/40
5,qwen2.5-14b,N0,reportes_en_filas,21/37,24/40,21/40
6,qwen2.5-14b,N1,senales_en_filas,27/37,30/40,27/40
7,qwen2.5-14b,N1,reportes_en_filas,27/37,30/40,27/40
8,qwen3.5-9b,N0,senales_en_filas,26/37,28/40,27/40
9,qwen3.5-9b,N0,reportes_en_filas,22/37,25/40,22/40


In [27]:
# --- 4.5 Por ventana: referencia y veredicto de cada modelo y variante (1 = anomalía, · = sin veredicto) ---
df["columna"] = df["modelo"] + " | " + df["metodo"] + " | " + df["orientacion"].str.replace("_en_filas", "")
por_ventana = (df.assign(celda=df["veredicto"].map(lambda v: "·" if pd.isna(v) else str(int(v))))
                 .pivot(index="id", columns="columna", values="celda"))
por_ventana = df.drop_duplicates("id").set_index("id")[["grupo", "referencia"]].join(por_ventana)
with pd.option_context("display.max_rows", 100, "display.max_columns", 50):
    display(por_ventana)

,grupo,referencia,qwen2.5-14b | N0 | reportes,qwen2.5-14b | N0 | senales,qwen2.5-14b | N1 | reportes,qwen2.5-14b | N1 | senales,qwen2.5-7b | N0 | reportes,qwen2.5-7b | N0 | senales,qwen2.5-7b | N1 | reportes,qwen2.5-7b | N1 | senales,qwen3.5-9b | N0 | reportes,qwen3.5-9b | N0 | senales,qwen3.5-9b | N1 | reportes,qwen3.5-9b | N1 | senales
id,,,,,,,,,,,,,,
V01,control_azar,normal,1,0,0,1,1,1,1,1,1,0,1,1
V02,control_alta_actividad,anomalia,0,0,0,1,0,1,0,1,0,0,1,1
V03,alerta_consenso_comun,anomalia,1,1,1,1,1,1,1,1,1,1,1,1
V04,alerta_consenso_comun,anomalia,1,1,1,1,1,1,1,1,1,1,1,1
V05,alerta_consenso_comun,anomalia,1,1,1,1,1,1,1,1,1,0,1,1
V06,control_azar,normal,0,1,1,1,1,1,1,1,1,1,1,1
V07,control_azar,normal,1,1,0,0,1,1,0,1,1,1,1,1
V08,control_alta_actividad,anomalia_mitigada,0,0,1,1,1,0,1,1,0,0,1,0
V09,alerta_consenso_comun,anomalia,1,0,1,1,0,0,1,1,1,0,1,1


## 5. Revisión de las respuestas

- **5.1:** una ventana: la gráfica con su tabla (nombres cortos) y el veredicto y la probabilidad de cada modelo y variante.
- **5.2:** el informe HTML con todas las ventanas y **las respuestas completas del modelo**. Lleva los nombres reales: se guarda en `privado/experimento_2/informes/`. Descárgalo de Drive y ábrelo en el navegador (funciona sin conexión).

In [28]:
# --- 5.1 Una ventana: gráfica + tabla y el veredicto de cada modelo y variante ---
import plotly.io as pio
from visualizacion import graficar_ventana_y_tabla

pio.renderers.default = "vscode"   # "colab" si lo ejecutas en Colab desde el navegador
ID_VENTANA = VENTANAS[0]["id"]     # cambia el identificador para ver otra

w = next(v for v in VENTANAS if v["id"] == ID_VENTANA)
graficar_ventana_y_tabla(fuente.ventana(w["conjunto"], w["indice"]), NOMBRES_CONTADORES,
                         f"{ID_VENTANA} · referencia: {w['referencia']} · grupo: {w['grupo']}")
display(exp2.detalle_ventana(df, ID_VENTANA)[["modelo", "metodo", "orientacion", "veredicto", "p_interna"]])

,modelo,metodo,orientacion,veredicto,p_interna
161,qwen2.5-14b,N0,reportes_en_filas,1,1.000000
1,qwen2.5-7b,N0,reportes_en_filas,1,0.999996
321,qwen3.5-9b,N0,reportes_en_filas,1,0.987568
160,qwen2.5-14b,N0,senales_en_filas,0,0.027347
0,qwen2.5-7b,N0,senales_en_filas,1,1.000000
320,qwen3.5-9b,N0,senales_en_filas,0,0.000140
163,qwen2.5-14b,N1,reportes_en_filas,0,0.000005
3,qwen2.5-7b,N1,reportes_en_filas,1,0.999989
323,qwen3.5-9b,N1,reportes_en_filas,1,0.997199
162,qwen2.5-14b,N1,senales_en_filas,1,1.000000


In [29]:
# --- 5.2 Informe HTML con las respuestas del modelo (privado) ---
correspondencia = leer_correspondencia()
nombres_reales = [correspondencia[n] for n in NOMBRES_CONTADORES]   # los que vio el modelo
os.makedirs(RUTA_INFORMES, exist_ok=True)
valores = {w["id"]: fuente.ventana(w["conjunto"], w["indice"]) for w in VENTANAS}
ruta_informe = exp2.informe_html(df, valores, nombres_reales,
                                 f"{RUTA_INFORMES}/informe_{CONJUNTO}_{datetime.now():%Y%m%d_%H%M}.html",
                                 f"Experimento 2 · {CONJUNTO} · respuestas por ventana")
print("Informe (privado):", ruta_informe)

Informe (privado): /content/drive/MyDrive/TFM_VIU_2026/datos/privado/experimento_2/informes/informe_test_20261013_1850.html


## 6. Guardado

En `resultados/experimento_2/<conjunto>/`: las tablas en CSV y un JSON con todo lo anterior y una fila por llamada (sin textos del modelo). Es lo que se puede compartir.

In [30]:
# --- 6. Cifras (sin textos del modelo) en resultados/experimento_2 ---
carpeta = f"{RUTA_EXPERIMENTO_2}/{CONJUNTO}"
os.makedirs(carpeta, exist_ok=True)
tablas = {"comparacion_principal": tabla_principal, "metricas": tabla_metricas, "grupos": tabla_grupos,
          "n0_frente_a_n1": tabla_n0_n1, "pares_de_modelos": tabla_pares, "dudosas": tabla_dudosas}
for nombre, tabla in tablas.items():
    tabla.to_csv(f"{carpeta}/{nombre}.csv", index=False)

registros = lambda t: json.loads(t.to_json(orient="records"))   # NaN -> null y tipos de numpy -> JSON
COLUMNAS = ["modelo", "id", "metodo", "orientacion", "veredicto", "p_interna", "json_valido", "cortada",
            "referencia", "grupo"]
resumen = {"conjunto": CONJUNTO, "fecha": datetime.now().strftime("%Y-%m-%d %H:%M"),
           "ejecuciones": {"qwen2.5-7b": os.path.basename(REFERENCIA_QWEN25[CONJUNTO])[:-4], **ejecuciones},
           "comparacion_principal_fijada": PRINCIPAL, **{k: registros(t) for k, t in tablas.items()},
           "llamadas": registros(df[COLUMNAS])}
ruta = f"{carpeta}/comparacion_{CONJUNTO}_{datetime.now():%Y%m%d_%H%M}.json"
with open(ruta, "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=1)
print("Guardado en:", carpeta, "| resumen para compartir:", os.path.basename(ruta))

Guardado en: /content/drive/MyDrive/TFM_VIU_2026/datos/resultados/experimento_2/test | resumen para compartir: comparacion_test_20261013_1850.json
